# WiDS 2026 Wildfire Survival Mega Ensemble


In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import re
import math
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from scipy.special import expit, logit

warnings.filterwarnings("ignore")

SEEDS = [2026, 3407, 7777, 9901]
N_FOLDS = 5
HORIZONS = [12, 24, 48, 72]
INTERVALS = [(0, 12), (12, 24), (24, 48), (48, 72)]

KNOWN_ARCHIVE_SCORES = {
    1: 0.96617, 2: 0.96540, 3: 0.96961, 4: 0.97252, 5: 0.97167,
    6: 0.97204, 7: 0.97252, 8: 0.97058, 9: 0.95804, 10: 0.97118,
    11: 0.96933, 12: 0.96539, 13: 0.96400, 14: 0.95893, 15: 0.96617,
    16: 0.96539, 17: 0.97252, 18: 0.96002, 19: 0.95628, 20: 0.96216,
    21: 0.95357, 22: 0.97252, 23: 0.97191,
}

random.seed(2026)
np.random.seed(2026)

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

try:
    import xgboost as xgb
except Exception:
    xgb = None


def find_file(name_candidates):
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    found = []
    for root in search_roots:
        if root.exists():
            for name in name_candidates:
                for p in root.rglob(name):
                    found.append(p)
    if not found:
        raise FileNotFoundError(f'Could not find any of: {name_candidates}')
    def score_path(p):
        s = str(p).lower()
        score = 0
        if 'wids' in s:
            score += 10
        if 'anonymous-contest' in s:
            score += 7
        if 'globaldatathon' in s or 'globaldathon' in s:
            score += 6
        if 'working' in s:
            score += 2
        return (-score, len(str(p)))
    found = sorted(set(found), key=score_path)
    return found[0]


def load_inputs():
    train_path = find_file(['train.csv'])
    test_path = find_file(['test.csv'])
    try:
        sample_path = find_file(['sample_submission.csv'])
    except Exception:
        sample_path = None
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    sample = pd.read_csv(sample_path) if sample_path is not None else pd.DataFrame({'event_id': test['event_id']})
    return train, test, sample, train_path, test_path, sample_path


def to_numeric_frame(df):
    out = df.copy()
    for c in out.columns:
        if c != 'event_id':
            out[c] = pd.to_numeric(out[c], errors='coerce')
    return out


def add_engineered_features(df):
    X = df.copy()
    def g(col, default=0.0):
        if col in X.columns:
            return X[col].astype(float)
        return pd.Series(default, index=X.index, dtype=float)

    pi = math.pi
    dist = g('dist_min_ci_0_5h')
    area0 = np.clip(g('area_first_ha'), 0, None)
    growth_abs = np.clip(g('area_growth_abs_0_5h'), 0, None)
    growth_rate_ha = g('area_growth_rate_ha_per_h')
    radial_rate = g('radial_growth_rate_m_per_h')
    closing = g('closing_speed_m_per_h')
    along = g('along_track_speed')
    centroid_speed = g('centroid_speed_m_per_h')
    projected_advance = g('projected_advance_m')
    alignment_abs = np.clip(g('alignment_abs'), 0, 1)
    cross_track = g('cross_track_component')
    dist_slope = g('dist_slope_ci_0_5h')
    dist_change = g('dist_change_ci_0_5h')
    dist_accel = g('dist_accel_m_per_h2')
    dist_r2 = np.clip(g('dist_fit_r2_0_5h'), 0, 1)
    low_res = np.clip(g('low_temporal_resolution_0_5h'), 0, 1)
    num_perim = np.clip(g('num_perimeters_0_5h'), 0, None)
    dt = np.clip(g('dt_first_last_0_5h'), 0, None)
    hour = g('event_start_hour')
    dow = g('event_start_dayofweek')
    month = g('event_start_month')

    X['dist_min_km'] = dist / 1000.0
    X['log_dist_min'] = np.log1p(np.clip(dist, 0, None))
    X['margin_to_5km_m'] = dist - 5000.0
    X['dist_inv'] = 1.0 / (1000.0 + np.clip(dist, 0, None))

    X['closing_pos_mph'] = np.maximum(0, closing)
    X['closing_neg_mph'] = np.maximum(0, -closing)
    X['toward_pos_mph'] = np.maximum(0, along)
    X['radial_pos_mph'] = np.maximum(0, radial_rate)
    X['front_speed_mph'] = X['closing_pos_mph'] + 0.70 * X['toward_pos_mph'] + 0.55 * X['radial_pos_mph']
    X['front_speed_alt_mph'] = np.maximum(0, -dist_slope) + 0.40 * np.maximum(0, centroid_speed * alignment_abs) + 0.40 * X['radial_pos_mph']

    X['eta_closing_h'] = np.where(X['closing_pos_mph'] > 1.0, X['margin_to_5km_m'] / (X['closing_pos_mph'] + 1.0), 999.0)
    X['eta_front_h'] = np.where(X['front_speed_mph'] > 1.0, X['margin_to_5km_m'] / (X['front_speed_mph'] + 1.0), 999.0)
    X['eta_front_alt_h'] = np.where(X['front_speed_alt_mph'] > 1.0, X['margin_to_5km_m'] / (X['front_speed_alt_mph'] + 1.0), 999.0)

    X['near_miss_margin_m'] = X['margin_to_5km_m'] - projected_advance - 8.0 * X['radial_pos_mph']
    X['threat_gravity'] = (np.log1p(area0) + np.log1p(growth_abs)) * (1.0 + alignment_abs) * (1.0 + X['front_speed_mph'] / 250.0) * X['dist_inv']
    X['motion_pressure'] = np.log1p(np.abs(centroid_speed)) * (0.6 + 0.4 * alignment_abs)
    X['growth_pressure'] = np.log1p(np.maximum(0, growth_rate_ha)) * (1.0 + X['radial_pos_mph'] / 250.0)
    X['quality_weight'] = (1.0 + np.log1p(num_perim)) * (1.0 + np.minimum(dt, 5.0)) * (1.0 - 0.45 * low_res)
    X['trend_confidence'] = (1.0 - 0.5 * low_res) * (0.35 + 0.65 * dist_r2)
    X['aligned_closing'] = X['closing_pos_mph'] * alignment_abs
    X['crosswind_ratio'] = np.abs(cross_track) / (1.0 + np.abs(along))
    X['advance_frac'] = projected_advance / (50.0 + np.clip(dist, 0, None))
    X['mass_over_dist'] = (np.log1p(area0) + 0.7 * np.log1p(growth_abs)) / (1.0 + X['dist_min_km'])
    X['closing_x_r2'] = X['closing_pos_mph'] * dist_r2
    X['dist_change_abs'] = np.abs(dist_change)
    X['dist_accel_abs'] = np.abs(dist_accel)
    X['speed_alignment_mix'] = np.maximum(0, centroid_speed) * (0.5 + 0.5 * alignment_abs)
    X['spread_quality'] = (1.0 - low_res) * (1.0 + np.log1p(num_perim)) * (0.5 + 0.5 * dist_r2)
    X['growth_to_dist'] = np.log1p(np.maximum(0, growth_rate_ha)) / (1.0 + X['dist_min_km'])
    X['area_to_dist'] = np.log1p(area0) / (1.0 + X['dist_min_km'])
    X['instability'] = np.abs(dist_accel) * (1.0 + np.abs(cross_track) / 1000.0)
    X['closing_margin_ratio'] = X['margin_to_5km_m'] / (50.0 + X['front_speed_mph'])
    X['threat_momentum'] = (1.0 + alignment_abs) * (X['front_speed_mph'] + np.maximum(0, centroid_speed)) * X['dist_inv']

    for H in HORIZONS:
        X[f'margin_front_{H}h'] = X['margin_to_5km_m'] - H * X['front_speed_mph']
        X[f'margin_alt_{H}h'] = X['margin_to_5km_m'] - H * X['front_speed_alt_mph']
        X[f'margin_closing_{H}h'] = X['margin_to_5km_m'] - H * X['closing_pos_mph']
        X[f'eta_front_le_{H}h'] = (X['eta_front_h'] <= H).astype(float)
        X[f'eta_alt_le_{H}h'] = (X['eta_front_alt_h'] <= H).astype(float)
        X[f'near_flag_{H}h'] = (X[f'margin_front_{H}h'] <= 0).astype(float)
        X[f'gravity_over_margin_{H}h'] = X['threat_gravity'] / (100.0 + np.abs(X[f'margin_front_{H}h']))

    if 'spread_bearing_deg' in X.columns:
        rad = np.deg2rad(X['spread_bearing_deg'].astype(float))
        X['bearing_sin_deg'] = np.sin(rad)
        X['bearing_cos_deg'] = np.cos(rad)

    if 'event_start_hour' in X.columns:
        X['hour_sin'] = np.sin(2.0 * pi * hour / 24.0)
        X['hour_cos'] = np.cos(2.0 * pi * hour / 24.0)
    if 'event_start_dayofweek' in X.columns:
        X['dow_sin'] = np.sin(2.0 * pi * dow / 7.0)
        X['dow_cos'] = np.cos(2.0 * pi * dow / 7.0)
    if 'event_start_month' in X.columns:
        X['month_sin'] = np.sin(2.0 * pi * (month - 1.0) / 12.0)
        X['month_cos'] = np.cos(2.0 * pi * (month - 1.0) / 12.0)

    return X


def clip_extremes(train_df, test_df, feature_cols, low_q=0.003, high_q=0.997):
    train_df = train_df.copy()
    test_df = test_df.copy()
    for c in feature_cols:
        s = pd.to_numeric(train_df[c], errors='coerce')
        if s.notna().sum() < 20:
            continue
        lo = s.quantile(low_q)
        hi = s.quantile(high_q)
        if pd.isna(lo) or pd.isna(hi) or hi <= lo:
            continue
        train_df[c] = train_df[c].clip(lo, hi)
        test_df[c] = test_df[c].clip(lo, hi)
    return train_df, test_df


def repair_monotonic(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 0.0, 1.0)
    p = np.maximum.accumulate(p, axis=1)
    p = np.clip(p, 0.0, 1.0)
    return p


def brier_at_horizon(time, event, pred, horizon):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    pred = np.asarray(pred, dtype=float)
    mask = ~((event == 0) & (time <= horizon))
    if mask.sum() == 0:
        return 0.25
    y = ((event == 1) & (time <= horizon)).astype(float)
    return float(np.mean((pred[mask] - y[mask]) ** 2))


def weighted_brier(time, event, pred):
    pred = repair_monotonic(pred)
    b24 = brier_at_horizon(time, event, pred[:, 1], 24)
    b48 = brier_at_horizon(time, event, pred[:, 2], 48)
    b72 = brier_at_horizon(time, event, pred[:, 3], 72)
    return 0.3 * b24 + 0.4 * b48 + 0.3 * b72


def risk_from_probs(pred):
    pred = repair_monotonic(pred)
    p12, p24, p48, p72 = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3]
    risk_lb = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    q1 = np.clip(p12, 0, 1)
    q2 = np.clip(p24 - p12, 0, 1)
    q3 = np.clip(p48 - p24, 0, 1)
    q4 = np.clip(p72 - p48, 0, 1)
    q5 = np.clip(1.0 - p72, 0, 1)
    expected_time = 6.0 * q1 + 18.0 * q2 + 36.0 * q3 + 60.0 * q4 + 72.0 * q5
    risk_time = 1.0 - expected_time / 72.0
    return 0.70 * risk_lb + 0.30 * risk_time


def c_index(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    risk = np.asarray(risk, dtype=float)
    n = len(time)
    concordant = 0.0
    comparable = 0.0
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if time[i] < time[j]:
                comparable += 1.0
                if risk[i] > risk[j]:
                    concordant += 1.0
                elif risk[i] == risk[j]:
                    concordant += 0.5
    if comparable == 0:
        return 0.5
    return concordant / comparable


def hybrid_score(time, event, pred):
    pred = repair_monotonic(pred)
    wb = weighted_brier(time, event, pred)
    ci = c_index(time, event, risk_from_probs(pred))
    return 0.3 * ci + 0.7 * (1.0 - wb)


def horizon_target(event, time, horizon):
    event = np.asarray(event, dtype=int)
    time = np.asarray(time, dtype=float)
    eligible = ~((event == 0) & (time <= horizon))
    y = ((event == 1) & (time <= horizon)).astype(int)
    return eligible, y


def interval_target(event, time, start_h, end_h):
    event = np.asarray(event, dtype=int)
    time = np.asarray(time, dtype=float)
    at_risk = time > start_h
    eligible = at_risk & ~((event == 0) & (time <= end_h))
    y = ((event == 1) & (time > start_h) & (time <= end_h)).astype(int)
    return eligible, y


def make_strat_labels(time, event):
    labels = []
    for t, e in zip(time, event):
        if e == 1:
            if t <= 12:
                lab = 'hit12'
            elif t <= 24:
                lab = 'hit24'
            elif t <= 48:
                lab = 'hit48'
            else:
                lab = 'hit72'
        else:
            if t <= 24:
                lab = 'cens24'
            elif t <= 48:
                lab = 'cens48'
            else:
                lab = 'cens72'
        labels.append(lab)
    labels = pd.Series(labels).astype(str)
    counts = labels.value_counts()
    rare = set(counts[counts < N_FOLDS].index)
    labels = labels.where(~labels.isin(rare), other=pd.Series(event).map({0: 'cens', 1: 'hit'}))
    return labels.astype(str).values


def get_cv_splits(time, event):
    strat = make_strat_labels(time, event)
    splits = []
    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for tr_idx, va_idx in skf.split(np.zeros(len(strat)), strat):
            splits.append((tr_idx, va_idx, seed))
    return splits


def fit_isotonic_or_constant(scores, y):
    scores = np.asarray(scores, dtype=float)
    y = np.asarray(y, dtype=int)
    mask = np.isfinite(scores)
    scores = scores[mask]
    y = y[mask]
    base = float(np.mean(y)) if len(y) else 0.5
    if len(y) < 16 or len(np.unique(y)) < 2 or len(np.unique(scores)) < 4:
        return lambda z, base=base: np.full(len(np.asarray(z)), base, dtype=float)
    ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
    ir.fit(scores, y)
    return lambda z, ir=ir, base=base: np.clip(0.90 * ir.predict(np.asarray(z, dtype=float)) + 0.10 * base, 0.0, 1.0)


def safe_constant(y):
    y = np.asarray(y)
    if len(y) == 0:
        return 0.5
    return float(np.clip(np.mean(y), 1e-4, 1.0 - 1e-4))


def make_linear_pipeline(seed, mode='logistic', n_neighbors=11):
    if mode == 'logistic':
        model = LogisticRegression(C=0.7, solver='liblinear', max_iter=2000, random_state=seed)
    else:
        model = KNeighborsRegressor(n_neighbors=max(3, n_neighbors), weights='distance', metric='minkowski', p=2)
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler(with_mean=True, with_std=True)),
        ('model', model),
    ])


def fit_tree_binary_ensemble(X_tr, y_tr, X_va, X_te, seed):
    y_tr = np.asarray(y_tr, dtype=int)
    p_base = safe_constant(y_tr)
    preds_val = []
    preds_test = []
    pos = int(y_tr.sum())
    neg = int(len(y_tr) - pos)
    if pos == 0 or neg == 0:
        return np.full(len(X_va), p_base, dtype=float), np.full(len(X_te), p_base, dtype=float)
    spw = float(np.sqrt(max(neg, 1) / max(pos, 1)))

    if CatBoostClassifier is not None:
        for depth, iters, lr in [(4, 500, 0.045), (6, 700, 0.030)]:
            try:
                model = CatBoostClassifier(
                    loss_function='Logloss',
                    eval_metric='Logloss',
                    depth=depth,
                    learning_rate=lr,
                    iterations=iters,
                    l2_leaf_reg=6.0,
                    random_strength=0.8,
                    bootstrap_type='Bayesian',
                    bagging_temperature=1.0,
                    class_weights=[1.0, spw],
                    random_seed=seed + depth + iters,
                    verbose=False,
                    allow_writing_files=False,
                )
                model.fit(X_tr, y_tr, verbose=False)
                preds_val.append(model.predict_proba(X_va)[:, 1])
                preds_test.append(model.predict_proba(X_te)[:, 1])
            except Exception:
                pass

    if LGBMClassifier is not None:
        for leaves, n_est, lr in [(15, 450, 0.040), (31, 700, 0.028)]:
            try:
                model = LGBMClassifier(
                    objective='binary',
                    n_estimators=n_est,
                    learning_rate=lr,
                    num_leaves=leaves,
                    max_depth=-1,
                    min_child_samples=8,
                    subsample=0.82,
                    subsample_freq=1,
                    colsample_bytree=0.72,
                    reg_alpha=0.10,
                    reg_lambda=1.8,
                    random_state=seed + leaves + n_est,
                    n_jobs=-1,
                    class_weight={0: 1.0, 1: spw},
                    verbose=-1,
                )
                model.fit(X_tr, y_tr)
                preds_val.append(model.predict_proba(X_va)[:, 1])
                preds_test.append(model.predict_proba(X_te)[:, 1])
            except Exception:
                pass

    if xgb is not None:
        for md, n_est, lr in [(3, 350, 0.045), (4, 500, 0.032)]:
            try:
                model = xgb.XGBClassifier(
                    objective='binary:logistic',
                    eval_metric='logloss',
                    n_estimators=n_est,
                    learning_rate=lr,
                    max_depth=md,
                    min_child_weight=2,
                    subsample=0.85,
                    colsample_bytree=0.75,
                    reg_lambda=1.6,
                    reg_alpha=0.0,
                    gamma=0.0,
                    tree_method='hist',
                    random_state=seed + md + n_est,
                    n_jobs=-1,
                    scale_pos_weight=spw,
                )
                model.fit(X_tr, y_tr)
                preds_val.append(model.predict_proba(X_va)[:, 1])
                preds_test.append(model.predict_proba(X_te)[:, 1])
            except Exception:
                pass

    if not preds_val:
        return np.full(len(X_va), p_base, dtype=float), np.full(len(X_te), p_base, dtype=float)
    return np.mean(preds_val, axis=0), np.mean(preds_test, axis=0)


def fit_analog_ensemble(X_tr, y_tr, X_va, X_te, seed):
    y_tr = np.asarray(y_tr, dtype=int)
    p_base = safe_constant(y_tr)
    if len(np.unique(y_tr)) < 2 or len(X_tr) < 8:
        return np.full(len(X_va), p_base, dtype=float), np.full(len(X_te), p_base, dtype=float)
    preds_val = []
    preds_test = []

    try:
        pipe = make_linear_pipeline(seed, mode='logistic')
        pipe.fit(X_tr, y_tr)
        preds_val.append(pipe.predict_proba(X_va)[:, 1])
        preds_test.append(pipe.predict_proba(X_te)[:, 1])
    except Exception:
        pass

    try:
        k = int(max(5, min(17, round(len(X_tr) ** 0.5) * 2 - 1)))
        pipe = make_linear_pipeline(seed, mode='knn', n_neighbors=k)
        pipe.fit(X_tr, y_tr)
        preds_val.append(np.clip(pipe.predict(X_va), 0.0, 1.0))
        preds_test.append(np.clip(pipe.predict(X_te), 0.0, 1.0))
    except Exception:
        pass

    if not preds_val:
        return np.full(len(X_va), p_base, dtype=float), np.full(len(X_te), p_base, dtype=float)
    return np.clip(np.mean(preds_val, axis=0), 0.0, 1.0), np.clip(np.mean(preds_test, axis=0), 0.0, 1.0)


def direct_family(X_tree, X_analog, X_test_tree, X_test_analog, time, event, splits):
    n = len(X_tree)
    m = len(X_test_tree)
    oof = np.zeros((n, len(HORIZONS)), dtype=float)
    oof_n = np.zeros((n, len(HORIZONS)), dtype=float)
    pred = np.zeros((m, len(HORIZONS)), dtype=float)
    pred_n = np.zeros((m, len(HORIZONS)), dtype=float)

    for h_idx, H in enumerate(HORIZONS):
        eligible_all, y_all = horizon_target(event, time, H)
        for tr_idx, va_idx, seed in splits:
            tr_fit = tr_idx[eligible_all[tr_idx]]
            if len(tr_fit) < 8:
                p_const = safe_constant(y_all[tr_fit] if len(tr_fit) else y_all[tr_idx])
                oof[va_idx, h_idx] += p_const
                oof_n[va_idx, h_idx] += 1.0
                pred[:, h_idx] += p_const
                pred_n[:, h_idx] += 1.0
                continue
            pv_tree, pt_tree = fit_tree_binary_ensemble(X_tree.iloc[tr_fit], y_all[tr_fit], X_tree.iloc[va_idx], X_test_tree, seed + H)
            pv_analog, pt_analog = fit_analog_ensemble(X_analog.iloc[tr_fit], y_all[tr_fit], X_analog.iloc[va_idx], X_test_analog, seed + 10 + H)
            pv = 0.78 * pv_tree + 0.22 * pv_analog
            pt = 0.78 * pt_tree + 0.22 * pt_analog
            oof[va_idx, h_idx] += np.clip(pv, 0.0, 1.0)
            oof_n[va_idx, h_idx] += 1.0
            pred[:, h_idx] += np.clip(pt, 0.0, 1.0)
            pred_n[:, h_idx] += 1.0

    oof = np.divide(oof, np.maximum(oof_n, 1.0))
    pred = np.divide(pred, np.maximum(pred_n, 1.0))
    return repair_monotonic(oof), repair_monotonic(pred)


def hazard_family(X_tree, X_analog, X_test_tree, X_test_analog, time, event, splits):
    n = len(X_tree)
    m = len(X_test_tree)
    oof_h = np.zeros((n, len(INTERVALS)), dtype=float)
    oof_n = np.zeros((n, len(INTERVALS)), dtype=float)
    pred_h = np.zeros((m, len(INTERVALS)), dtype=float)
    pred_n = np.zeros((m, len(INTERVALS)), dtype=float)

    for j, (start_h, end_h) in enumerate(INTERVALS):
        eligible_all, y_all = interval_target(event, time, start_h, end_h)
        for tr_idx, va_idx, seed in splits:
            tr_fit = tr_idx[eligible_all[tr_idx]]
            if len(tr_fit) < 8:
                p_const = safe_constant(y_all[tr_fit] if len(tr_fit) else y_all[tr_idx])
                oof_h[va_idx, j] += p_const
                oof_n[va_idx, j] += 1.0
                pred_h[:, j] += p_const
                pred_n[:, j] += 1.0
                continue
            pv_tree, pt_tree = fit_tree_binary_ensemble(X_tree.iloc[tr_fit], y_all[tr_fit], X_tree.iloc[va_idx], X_test_tree, seed + end_h)
            pv_analog, pt_analog = fit_analog_ensemble(X_analog.iloc[tr_fit], y_all[tr_fit], X_analog.iloc[va_idx], X_test_analog, seed + 10 + end_h)
            pv = 0.82 * pv_tree + 0.18 * pv_analog
            pt = 0.82 * pt_tree + 0.18 * pt_analog
            oof_h[va_idx, j] += np.clip(pv, 0.0, 1.0)
            oof_n[va_idx, j] += 1.0
            pred_h[:, j] += np.clip(pt, 0.0, 1.0)
            pred_n[:, j] += 1.0

    oof_h = np.divide(oof_h, np.maximum(oof_n, 1.0))
    pred_h = np.divide(pred_h, np.maximum(pred_n, 1.0))

    def hazards_to_cum(h):
        h = np.clip(h, 0.0, 1.0)
        p12 = h[:, 0]
        p24 = p12 + (1.0 - p12) * h[:, 1]
        p48 = p24 + (1.0 - p24) * h[:, 2]
        p72 = p48 + (1.0 - p48) * h[:, 3]
        return repair_monotonic(np.column_stack([p12, p24, p48, p72]))

    return hazards_to_cum(oof_h), hazards_to_cum(pred_h)


def survival_family(X_tree, X_test_tree, time, event, splits):
    n = len(X_tree)
    m = len(X_test_tree)
    oof = np.zeros((n, len(HORIZONS)), dtype=float)
    oof_n = np.zeros((n, len(HORIZONS)), dtype=float)
    pred = np.zeros((m, len(HORIZONS)), dtype=float)
    pred_n = np.zeros((m, len(HORIZONS)), dtype=float)

    if xgb is None:
        base = np.zeros((n, len(HORIZONS)), dtype=float)
        p = np.zeros((m, len(HORIZONS)), dtype=float)
        for i, H in enumerate(HORIZONS):
            _, y = horizon_target(event, time, H)
            base[:, i] = safe_constant(y)
            p[:, i] = safe_constant(y)
        return repair_monotonic(base), repair_monotonic(p)

    for tr_idx, va_idx, seed in splits:
        X_tr = X_tree.iloc[tr_idx]
        X_va = X_tree.iloc[va_idx]
        X_te = X_test_tree
        lower = np.asarray(time[tr_idx], dtype=float)
        upper = np.where(np.asarray(event[tr_idx], dtype=int) == 1, lower, np.inf)
        risk_train_list = []
        risk_val_list = []
        risk_test_list = []

        try:
            dtr = xgb.DMatrix(X_tr, missing=np.nan)
            dva = xgb.DMatrix(X_va, missing=np.nan)
            dte = xgb.DMatrix(X_te, missing=np.nan)
            dtr.set_float_info('label_lower_bound', lower)
            dtr.set_float_info('label_upper_bound', upper)
            aft_params = {
                'objective': 'survival:aft',
                'eval_metric': 'aft-nloglik',
                'tree_method': 'hist',
                'learning_rate': 0.035,
                'max_depth': 3,
                'min_child_weight': 2,
                'subsample': 0.90,
                'colsample_bytree': 0.78,
                'lambda': 1.6,
                'alpha': 0.0,
                'aft_loss_distribution': 'logistic',
                'aft_loss_distribution_scale': 1.2,
                'verbosity': 0,
                'seed': seed + 111,
            }
            aft = xgb.train(aft_params, dtr, num_boost_round=280)
            risk_train_list.append(-aft.predict(dtr))
            risk_val_list.append(-aft.predict(dva))
            risk_test_list.append(-aft.predict(dte))
        except Exception:
            pass

        try:
            cox_label = np.where(np.asarray(event[tr_idx], dtype=int) == 1, np.asarray(time[tr_idx], dtype=float), -np.asarray(time[tr_idx], dtype=float))
            dtr = xgb.DMatrix(X_tr, label=cox_label, missing=np.nan)
            dva = xgb.DMatrix(X_va, missing=np.nan)
            dte = xgb.DMatrix(X_te, missing=np.nan)
            cox_params = {
                'objective': 'survival:cox',
                'eval_metric': 'cox-nloglik',
                'tree_method': 'hist',
                'learning_rate': 0.040,
                'max_depth': 3,
                'min_child_weight': 2,
                'subsample': 0.88,
                'colsample_bytree': 0.76,
                'lambda': 1.4,
                'alpha': 0.0,
                'verbosity': 0,
                'seed': seed + 222,
            }
            cox = xgb.train(cox_params, dtr, num_boost_round=260)
            risk_train_list.append(cox.predict(dtr))
            risk_val_list.append(cox.predict(dva))
            risk_test_list.append(cox.predict(dte))
        except Exception:
            pass

        if not risk_train_list:
            for i, H in enumerate(HORIZONS):
                _, y_tr_all = horizon_target(event[tr_idx], time[tr_idx], H)
                p_const = safe_constant(y_tr_all)
                oof[va_idx, i] += p_const
                oof_n[va_idx, i] += 1.0
                pred[:, i] += p_const
                pred_n[:, i] += 1.0
            continue

        risk_tr = np.mean(risk_train_list, axis=0)
        risk_va = np.mean(risk_val_list, axis=0)
        risk_te = np.mean(risk_test_list, axis=0)

        for i, H in enumerate(HORIZONS):
            elig_tr, y_tr_all = horizon_target(event[tr_idx], time[tr_idx], H)
            if elig_tr.sum() < 12:
                p_const = safe_constant(y_tr_all[elig_tr] if elig_tr.sum() > 0 else y_tr_all)
                pv = np.full(len(va_idx), p_const, dtype=float)
                pt = np.full(len(X_te), p_const, dtype=float)
            else:
                calibrator = fit_isotonic_or_constant(risk_tr[elig_tr], y_tr_all[elig_tr])
                pv = calibrator(risk_va)
                pt = calibrator(risk_te)
            oof[va_idx, i] += np.clip(pv, 0.0, 1.0)
            oof_n[va_idx, i] += 1.0
            pred[:, i] += np.clip(pt, 0.0, 1.0)
            pred_n[:, i] += 1.0

    oof = np.divide(oof, np.maximum(oof_n, 1.0))
    pred = np.divide(pred, np.maximum(pred_n, 1.0))
    return repair_monotonic(oof), repair_monotonic(pred)


def optimize_blend(time, event, family_oof):
    names = list(family_oof.keys())
    mats = [repair_monotonic(np.asarray(family_oof[n], dtype=float)) for n in names]
    k = len(mats)
    scores = {n: hybrid_score(time, event, m) for n, m in zip(names, mats)}
    best_pred = None
    best_cfg = None
    best_score = -1e18

    candidate_weights = []
    eye = np.eye(k)
    for i in range(k):
        candidate_weights.append(eye[i])
    candidate_weights.append(np.ones(k) / k)
    if k >= 2:
        sorted_names = sorted(names, key=lambda n: scores[n], reverse=True)
        top2 = [names.index(sorted_names[0]), names.index(sorted_names[1])]
        w = np.zeros(k)
        w[top2] = 0.5
        candidate_weights.append(w)

    alpha72_grid = [0.0, 0.10, 0.25, 0.40, 0.60, 0.80, 1.00]
    temp_grid = [0.88, 0.94, 1.00, 1.06, 1.12]

    def apply_cfg(weights, temp, alpha72):
        p = np.zeros_like(mats[0])
        for w, m in zip(weights, mats):
            p += w * m
        p = np.clip(p, 1e-6, 1.0 - 1e-6)
        p[:, :3] = expit(logit(p[:, :3]) / temp)
        p[:, 3] = (1.0 - alpha72) * p[:, 3] + alpha72 * 1.0
        return repair_monotonic(p)

    for w in candidate_weights:
        for temp in temp_grid:
            for alpha72 in alpha72_grid:
                p = apply_cfg(w, temp, alpha72)
                sc = hybrid_score(time, event, p)
                if sc > best_score:
                    best_score = sc
                    best_pred = p.copy()
                    best_cfg = {'weights': w.copy(), 'temp': temp, 'alpha72': alpha72}

    rng = np.random.default_rng(2026)
    for _ in range(5000):
        raw = rng.dirichlet(np.ones(k) * 1.15)
        temp = float(rng.uniform(0.87, 1.13))
        alpha72 = float(rng.choice(alpha72_grid))
        p = apply_cfg(raw, temp, alpha72)
        sc = hybrid_score(time, event, p)
        if sc > best_score:
            best_score = sc
            best_pred = p.copy()
            best_cfg = {'weights': raw.copy(), 'temp': temp, 'alpha72': alpha72}

    return best_pred, best_cfg, scores


def apply_blend(family_pred, cfg):
    names = list(family_pred.keys())
    mats = [repair_monotonic(np.asarray(family_pred[n], dtype=float)) for n in names]
    weights = np.asarray(cfg['weights'], dtype=float)
    p = np.zeros_like(mats[0])
    for w, m in zip(weights, mats):
        p += w * m
    p = np.clip(p, 1e-6, 1.0 - 1e-6)
    p[:, :3] = expit(logit(p[:, :3]) / float(cfg['temp']))
    p[:, 3] = (1.0 - float(cfg['alpha72'])) * p[:, 3] + float(cfg['alpha72']) * 1.0
    return repair_monotonic(p)


def load_archive_predictions(sample_ids):
    archive = {}
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    pattern = re.compile(r'samplecsv_sub(\d+)\.csv$', flags=re.I)
    files = []
    for root in roots:
        if root.exists():
            files.extend(root.rglob('samplecsv_sub*.csv'))
    seen = set()
    for fp in files:
        fp = Path(fp)
        if fp in seen:
            continue
        seen.add(fp)
        m = pattern.search(fp.name)
        if not m:
            continue
        sub_id = int(m.group(1))
        try:
            df = pd.read_csv(fp)
            need = ['event_id', 'prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']
            if not all(c in df.columns for c in need):
                continue
            df = df[need].copy().drop_duplicates('event_id')
            if set(df['event_id']) != set(sample_ids):
                continue
            df = df.set_index('event_id').loc[sample_ids].reset_index()
            p = repair_monotonic(df[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values)
            archive[sub_id] = {'path': str(fp), 'score': KNOWN_ARCHIVE_SCORES.get(sub_id, None), 'pred': p}
        except Exception:
            continue
    return archive


def build_archive_consensus(archive):
    if not archive:
        return None, {}
    rows = []
    for sub_id, meta in archive.items():
        score = meta['score'] if meta['score'] is not None else 0.965
        rows.append((sub_id, score, meta['pred']))
    rows = sorted(rows, key=lambda x: x[1], reverse=True)
    preds = [r[2] for r in rows]
    scores = np.array([r[1] for r in rows], dtype=float)
    best = scores.max()
    w_soft = np.exp((scores - best) / 0.0025)
    w_soft = w_soft / w_soft.sum()
    blend_soft = repair_monotonic(np.tensordot(w_soft, np.stack(preds, axis=0), axes=(0, 0)))
    topk = min(7, len(rows))
    blend_top = repair_monotonic(np.mean(np.stack([r[2] for r in rows[:topk]], axis=0), axis=0))
    if len(preds) >= 3:
        flat = np.stack([p.reshape(-1) for p in preds], axis=0)
        corr = np.corrcoef(flat)
        corr = np.nan_to_num(corr, nan=1.0, posinf=1.0, neginf=1.0)
        mean_corr = corr.mean(axis=1)
        div = np.clip(1.0 - mean_corr, 1e-6, None)
        w_div = (scores - scores.min() + 1e-4) * div
        w_div = w_div / w_div.sum()
        blend_div = repair_monotonic(np.tensordot(w_div, np.stack(preds, axis=0), axes=(0, 0)))
    else:
        blend_div = blend_top.copy()
    return {'archive_soft': blend_soft, 'archive_top': blend_top, 'archive_div': blend_div}, {'count': len(rows), 'top_scores': [float(x[1]) for x in rows[:min(10, len(rows))]]}


def validate_submission(sub, sample_ids):
    need = ['event_id', 'prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']
    assert list(sub.columns) == need, f'Bad columns: {sub.columns.tolist()}'
    assert sub['event_id'].is_unique, 'Duplicate event_id in submission'
    assert set(sub['event_id']) == set(sample_ids), 'event_id mismatch'
    assert sub.shape[0] == len(sample_ids), 'Row count mismatch'
    probs = sub[need[1:]].values
    assert np.isfinite(probs).all(), 'Non-finite probabilities'
    assert ((probs >= -1e-9) & (probs <= 1.0 + 1e-9)).all(), 'Probabilities out of bounds'
    assert np.all(probs[:, 0] <= probs[:, 1] + 1e-12), 'Non-monotonic 12->24'
    assert np.all(probs[:, 1] <= probs[:, 2] + 1e-12), 'Non-monotonic 24->48'
    assert np.all(probs[:, 2] <= probs[:, 3] + 1e-12), 'Non-monotonic 48->72'


train, test, sample, train_path, test_path, sample_path = load_inputs()
train = to_numeric_frame(train)
test = to_numeric_frame(test)
train_feat = add_engineered_features(train)
test_feat = add_engineered_features(test)

base_feature_cols = [c for c in train_feat.columns if c not in ['event_id', 'time_to_hit_hours', 'event'] and c in test_feat.columns]
base_feature_cols = [c for c in base_feature_cols if pd.to_numeric(train_feat[c], errors='coerce').notna().sum() > 0 and pd.to_numeric(train_feat[c], errors='coerce').nunique(dropna=True) > 1]
train_feat, test_feat = clip_extremes(train_feat, test_feat, base_feature_cols, low_q=0.003, high_q=0.997)

core_keywords = ['dist_min', 'margin_', 'eta_', 'threat_', 'front_speed', 'closing_', 'along_track_speed', 'alignment', 'projected_advance', 'radial_', 'growth_', 'area_', 'mass_', 'advance_frac', 'motion_pressure', 'trend_confidence', 'hour_', 'dow_', 'month_', 'quality', 'spread_bearing', 'crosswind', 'near_flag', 'gravity_over_margin']
core_cols = []
for c in base_feature_cols:
    if any(k in c for k in core_keywords):
        core_cols.append(c)
core_cols = list(dict.fromkeys(core_cols))
if len(core_cols) < 20:
    core_cols = base_feature_cols.copy()

X_tree = train_feat[base_feature_cols].copy()
X_test_tree = test_feat[base_feature_cols].copy()
X_analog = train_feat[core_cols].copy()
X_test_analog = test_feat[core_cols].copy()

time = train['time_to_hit_hours'].astype(float).values
event = train['event'].astype(int).values
splits = get_cv_splits(time, event)

oof_direct, pred_direct = direct_family(X_tree, X_analog, X_test_tree, X_test_analog, time, event, splits)
oof_hazard, pred_hazard = hazard_family(X_tree, X_analog, X_test_tree, X_test_analog, time, event, splits)
oof_surv, pred_surv = survival_family(X_tree, X_test_tree, time, event, splits)

family_oof = {
    'direct': oof_direct,
    'hazard': oof_hazard,
    'survival': oof_surv,
    'anchor': repair_monotonic(0.55 * oof_direct + 0.25 * oof_hazard + 0.20 * oof_surv),
}
family_pred = {
    'direct': pred_direct,
    'hazard': pred_hazard,
    'survival': pred_surv,
    'anchor': repair_monotonic(0.55 * pred_direct + 0.25 * pred_hazard + 0.20 * pred_surv),
}

oof_blend, blend_cfg, family_scores = optimize_blend(time, event, family_oof)
pred_blend = apply_blend(family_pred, blend_cfg)

sample_ids = sample['event_id'].tolist() if 'event_id' in sample.columns else test['event_id'].tolist()
no_archive_submission = pd.DataFrame({
    'event_id': sample_ids,
    'prob_12h': pred_blend[:, 0],
    'prob_24h': pred_blend[:, 1],
    'prob_48h': pred_blend[:, 2],
    'prob_72h': pred_blend[:, 3],
})
no_archive_submission = no_archive_submission.set_index('event_id').loc[sample_ids].reset_index()
validate_submission(no_archive_submission, sample_ids)
no_archive_submission.to_csv('submission_no_archive.csv', index=False)

archive_raw = load_archive_predictions(sample_ids)
archive_blends, archive_meta = build_archive_consensus(archive_raw)
main_pred = pred_blend.copy()
variant_files = {}

if archive_blends is not None:
    for name, arch_pred in archive_blends.items():
        for w in [0.08, 0.12, 0.18]:
            cand = repair_monotonic((1.0 - w) * pred_blend + w * arch_pred)
            fname = f'{name}_w{str(w).replace(".", "")}.csv'
            sub_tmp = pd.DataFrame({
                'event_id': sample_ids,
                'prob_12h': cand[:, 0],
                'prob_24h': cand[:, 1],
                'prob_48h': cand[:, 2],
                'prob_72h': cand[:, 3],
            })
            sub_tmp = sub_tmp.set_index('event_id').loc[sample_ids].reset_index()
            validate_submission(sub_tmp, sample_ids)
            sub_tmp.to_csv(fname, index=False)
            variant_files[fname] = {'blend': name, 'weight': w}
    archive_weight_main = 0.10 if archive_meta.get('count', 0) >= 3 else 0.06
    main_pred = repair_monotonic((1.0 - archive_weight_main) * pred_blend + archive_weight_main * archive_blends['archive_soft'])

submission = pd.DataFrame({
    'event_id': sample_ids,
    'prob_12h': main_pred[:, 0],
    'prob_24h': main_pred[:, 1],
    'prob_48h': main_pred[:, 2],
    'prob_72h': main_pred[:, 3],
})
submission = submission.set_index('event_id').loc[sample_ids].reset_index()
submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']] = repair_monotonic(submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values)
validate_submission(submission, sample_ids)
submission.to_csv('submission.csv', index=False)

report = {
    'train_path': str(train_path),
    'test_path': str(test_path),
    'sample_path': str(sample_path) if sample_path is not None else None,
    'n_train': int(train.shape[0]),
    'n_test': int(test.shape[0]),
    'n_features_tree': int(len(base_feature_cols)),
    'n_features_analog': int(len(core_cols)),
    'n_splits': int(len(splits)),
    'family_scores_local': {k: float(v) for k, v in family_scores.items()},
    'blend_cfg': {
        'weights': {name: float(w) for name, w in zip(list(family_oof.keys()), blend_cfg['weights'])},
        'temp': float(blend_cfg['temp']),
        'alpha72': float(blend_cfg['alpha72']),
    },
    'final_local_score': float(hybrid_score(time, event, oof_blend)),
    'archive_meta': archive_meta if archive_blends is not None else None,
    'variant_files': variant_files,
}
with open('blend_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print(submission.head(10).to_string(index=False))


{
  "train_path": "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv",
  "test_path": "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv",
  "sample_path": "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/sample_submission.csv",
  "n_train": 221,
  "n_test": 95,
  "n_features_tree": 103,
  "n_features_analog": 80,
  "n_splits": 20,
  "family_scores_local": {
    "direct": 0.9681861010330934,
    "hazard": 0.9666489381697294,
    "survival": 0.9636930794426461,
    "anchor": 0.9679139180865787
  },
  "blend_cfg": {
    "weights": {
      "direct": 0.49476630568011626,
      "hazard": 0.03301533452830412,
      "survival": 0.3545267320305537,
      "anchor": 0.11769162776102599
    },
    "temp": 0.8760964506987591,
    "alpha72": 0.25
  },
  "final_local_score": 0.9683248962771046,
  "archive_meta": null,
  "variant_files": {}
}
 event_id  prob_12h  prob_24h  prob_48h  prob_72h
 10662602  0.006251  0.009169  0.011501  0.999953
 13353600  0.688251  